# Linked Lists

**Seong Cho** · ACME Volume 2, Lab 1

**Lab objective:** begin the study of data structures by constructing a generic doubly linked list that behaves like a Python list, then using it to implement a few common restricted-access structures, ending with a deque.

<sub>Lab specification from the *Volume 2* lab manual of [Foundations of Applied Mathematics](https://foundations-of-applied-mathematics.github.io/) (BYU ACME), © Dr. J. Humpherys, used under [CC BY 3.0 US](http://creativecommons.org/licenses/by/3.0/us/). Problem statements are quoted from the spec; the notes are condensed from it. Solutions are my own.</sub>

In [1]:
# linked_lists.py
# """Volume 2: Linked Lists.
# <Name>
# <Class>
# <Date>
# """

## Nodes

Data structures organize data efficiently, and each kind has its own strengths and weaknesses: some take a long time to build but give quick access to their data, others build quickly but are slower to access. Python's built-in structures are `list`, `set`, `dict`, and `tuple`. This lab builds a structure that mimics the built-in `list` class with a different underlying implementation, so it is better than a plain `list` for some tasks and worse for others.

A *node* is like a standard-size box in a warehouse: lamps of different shapes are easier to store and retrieve once each one is boxed, and the data structure (the warehouse) only decides where and how the boxes are kept. A node class is usually simple: the data is stored as an attribute, and other attributes may be added (or inherited) for a particular data structure.

## Problem 1: Restricting node data

> **Problem 1.** Consider the following generic node class.
>
> ```python
> class Node:
>     """A basic node class for storing data."""
>     def __init__(self, data):
>         """Store 'data' in the 'value' attribute."""
>         self.value = data
> ```
>
> Modify the constructor so that it only accepts data of type `int`, `float`, or `str`. If another type of data is given, raise a `TypeError` with an appropriate error message. Modify the constructor docstring to document these restrictions.

> **Note.** Often the data stored in a node is actually a key value. The key might be a memory address, a dictionary key, or the index of an array where the true desired information resides. For simplicity, in this and the following lab we store actual data in node objects, not references to data located elsewhere.

In [2]:
# Problem 1
class Node:
    """A basic node class for storing data."""
    def __init__(self, data):
        """Store 'data' in the 'value' attribute. Raise TypeError if data
        isn't int, float, or string
        """
        if type(data) != int and type(data) != float and type(data) != str:
            raise TypeError("Data must be int, float, or string.")
        self.value = data

In [3]:
for data in [7, 3.5, "seven"]:
    print(f"Node({data!r}).value = {Node(data).value!r}")

for bad in [[1, 2], {1, 2}, 1 + 2j, None, (1,)]:
    try:
        Node(bad)
    except TypeError as e:
        print(f"Node({bad!r}) -> TypeError: {e}")
    else:
        raise AssertionError(f"Node accepted {bad!r}")

Node(7).value = 7
Node(3.5).value = 3.5
Node('seven').value = 'seven'
Node([1, 2]) -> TypeError: Data must be int, float, or string.
Node({1, 2}) -> TypeError: Data must be int, float, or string.
Node((1+2j)) -> TypeError: Data must be int, float, or string.
Node(None) -> TypeError: Data must be int, float, or string.
Node((1,)) -> TypeError: Data must be int, float, or string.


## Linked Lists

A *linked list* chains nodes together. It keeps a reference to the first node, the *head*, and often to the last node, the *tail*; each node stores a piece of data plus at least one reference to another node. Nodes of a *singly linked list* only reference the next node, so the list can only be traversed one way. Nodes of a *doubly linked list* also reference the previous node, so it can be traversed in both directions. The spec's node class for doubly linked lists inherits from `Node`:

In [4]:
class LinkedListNode(Node):
    """A node class for doubly linked lists. Inherits from the 'Node' class.
    Contains references to the next and previous nodes in the linked list.
    """
    def __init__(self, data):
        """Store 'data' in the 'value' attribute and initialize
        attributes for the next and previous nodes in the list.
        """
        Node.__init__(self, data)       # Use inheritance to set self.value.
        self.next = None
        self.prev = None

The spec's `LinkedList` class starts with an empty list (`head` and `tail` set to `None`) and an `append()` method with two cases: appending to an empty list (the new node becomes both the head and the tail) and appending to a nonempty list (link the new node after the tail in both directions, then reassign the tail). The full class, with the methods from Problems 2–5, is defined under Problem 2.

> **Achtung!** The `is` comparison operator is not the same as the `==` comparison operator. While `==` checks for numerical equality, `is` evaluates whether or not two objects are at the same location in memory.
>
> ```python
> # This evaluates to True since the numerical values are the same.
> >>> 7 == 7.0
> True
> # 7 is an int and 7.0 is a float, so they cannot be stored at the same
> # location in memory. Therefore 7 'is not' 7.0.
> >>> 7 is 7.0
> False
> ```
>
> For numerical comparisons, always use `==`. When comparing to built-in Python constants such as `None`, `True`, `False`, or `NotImplemented`, use `is` instead.

### find()

The class only keeps explicit track of the first and last nodes through `head` and `tail`. Every other node is reached by following each successive node's `next` and `prev` attributes.

## Problem 2: find()

> **Problem 2.** Add a method called `find(self, data)` to the `LinkedList` class that returns the first node in the list containing `data` (return the actual `LinkedListNode` object, not its value). If no such node exists, or if the list is empty, raise a `ValueError` with an appropriate error message.
>
> (Hint: if `current` is assigned to one of the nodes the list, what does the following line do?)
>
> ```python
> current = current.next
> ```

The whole `LinkedList` class is below. Besides the spec's `__init__()` and `append()` (with a `size` counter added for Problem 3), it holds `find()` (Problem 2), `__len__()` and `__str__()` (Problem 3), `remove()` (Problem 4), and `insert()` (Problem 5), each marked with a `# Problem N` comment.

In [5]:
class LinkedList:
    """Doubly linked list data structure class.

    Attributes:
        head (LinkedListNode): the first node in the list.
        tail (LinkedListNode): the last node in the list.
    """
    def __init__(self):
        """Initialize the 'head' and 'tail' attributes by setting
        them to 'None', since the list is empty initially.
        """
        self.head = None
        self.tail = None
        self.size = 0


    def append(self, data):
        """Append a new node containing 'data' to the end of the list."""
        # Create a new node to store the input data.
        new_node = LinkedListNode(data)
        if self.head is None:
            # If the list is empty, assign the head and tail attributes to
            # new_node, since it becomes the first and last node in the list.
            self.head = new_node
            self.tail = new_node
            self.size += 1
        else:
            # If the list is not empty, place new_node after the tail.
            self.tail.next = new_node               # tail --> new_node
            new_node.prev = self.tail               # tail <-- new_node
            # Now the last node in the list is new_node, so reassign the tail.
            self.tail = new_node
            self.size += 1

    # Problem 2
    def find(self, data):
        """Return the first node in the list containing 'data'.
        If no such node exists, raise a ValueError.

        Examples:
            >>> l = LinkedList()
            >>> for i in [1,3,5,7,9]:
            ...     l.append(i)
            ...
            >>> node = l.find(5)
            >>> node.value
            5
            >>> l.find(10)
            ValueError: <message>
        """
        if self.size == 0:
            raise ValueError("Data doesn't exist.")

        current = self.head
        while current.value != data:
            if current == self.tail:
                raise ValueError("Data doesn't exist.")
            current = current.next


        return current

    # Problem 3
    def __len__(self):
        """Return the number of nodes in the list.

        Examples:
            >>> l = LinkedList()
            >>> for i in [1,3,5]:
            ...     l.append(i)
            ...
            >>> len(l)
            3
            >>> l.append(7)
            >>> len(l)
            4
        """
        return self.size

    # Problem 3
    def __str__(self):
        """String representation: the same as a standard Python list.

        Examples:
            >>> l1 = LinkedList()   |   >>> l2 = LinkedList()
            >>> for i in [1,3,5]:   |   >>> for i in ['a','b',"c"]:
            ...     l1.append(i)    |   ...     l2.append(i)
            ...                     |   ...
            >>> print(l1)           |   >>> print(l2)
            [1, 3, 5]               |   ['a', 'b', 'c']
        """
        s = []
        if self.size == 0:
            return str(s)
        temp = self.head
        s.append(temp.value)
        while temp != self.tail:
            temp = temp.next
            s.append(temp.value)

        return str(s)

    # Problem 4
    def remove(self, data):
        """Remove the first node in the list containing 'data'. Return nothing.

        Raises:
            ValueError: if the list is empty, or does not contain 'data'.

        Examples:
            >>> print(l1)       |   >>> print(l2)
            [1, 3, 5, 7, 9]     |   [2, 4, 6, 8]
            >>> l1.remove(5)    |   >>> l2.remove(10)
            >>> l1.remove(1)    |   ValueError: <message>
            >>> l1.remove(9)    |   >>> l3 = LinkedList()
            >>> print(l1)       |   >>> l3.remove(10)
            [3, 7]              |   ValueError: <message>
        """
        if self.size == 0:
            raise ValueError("The list is empty.")
        temp = self.find(data)
        if temp == self.head:
            if self.size == 1:
                self.head = None
                self.tail = None
                self.size -= 1
                return
            self.head = temp.next
            self.head.prev = None
            self.size -= 1

        elif temp == self.tail:
            self.tail = temp.prev
            self.tail.next = None
            self.size -= 1

        else:
            temp.prev.next = temp.next
            temp.next.prev = temp.prev
            self.size -= 1

    # Problem 5
    def insert(self, data, place):
        """Insert a node containing 'data' immediately before the first node
        in the list containing 'place'. Return nothing.

        Raises:
            ValueError: if the list is empty, or does not contain 'place'.

        Examples:
            >>> print(l1)           |   >>> print(l1)
            [1, 3, 7]               |   [1, 3, 5, 7, 7]
            >>> l1.insert(7,7)      |   >>> l1.insert(3, 2)
            >>> print(l1)           |   ValueError: <message>
            [1, 3, 7, 7]            |
            >>> l1.insert(5,7)      |   >>> l2 = LinkedList()
            >>> print(l1)           |   >>> l2.insert(10,10)
            [1, 3, 5, 7, 7]         |   ValueError: <message>
        """
        if self.size == 0:
            raise ValueError("The list is empty.")

        new_node = LinkedListNode(data)
        temp = self.find(place)
        if temp == self.head:
            new_node.next = temp
            temp.prev = new_node
            self.head = new_node
            self.size += 1

        else:
            new_node.next = temp
            new_node.prev = temp.prev
            temp.prev.next = new_node
            temp.prev = new_node
            self.size += 1

In [6]:
# Accessing nodes through head, tail, next and prev (the spec's example).
my_list = LinkedList()
my_list.append(2)
my_list.append(4)
my_list.append(6)
print(my_list.head.value, my_list.head.next.value, my_list.head.next.next.value)
print(my_list.head.next.next is my_list.tail, my_list.tail.prev.prev is my_list.head)

# find() returns the node itself.
l = LinkedList()
for i in [1, 3, 5, 7, 9]:
    l.append(i)
node = l.find(5)
print(type(node).__name__, node.value)
assert node is l.head.next.next and l.find(1) is l.head and l.find(9) is l.tail

for lst, target in [(l, 10), (LinkedList(), 1)]:
    try:
        lst.find(target)
    except ValueError as e:
        print(f"find({target}) -> ValueError: {e}")
    else:
        raise AssertionError("find() should raise ValueError")

2 4 6
True True
LinkedListNode 5
find(10) -> ValueError: Data doesn't exist.
find(1) -> ValueError: Data doesn't exist.


## Magic Methods

Magic methods make a data structure much easier to use intuitively. At the bare minimum, the linked list should respond to built-in functions like `len()` and `print()` the way a Python list does.

## Problem 3: `__len__()` and `__str__()`

> **Problem 3.** Add magic methods to the `LinkedList` class so it behaves more like the built-in Python list.
>
> 1. Write the `__len__()` method so that the length of a `LinkedList` instance is equal to the number of nodes in the list. To accomplish this, consider adding an attribute that tracks the current size of the list. It should be updated every time a node is successfully added or removed.
> 2. Write the `__str__()` method so that when a `LinkedList` instance is printed, its output matches that of a Python list. Entries are separated by a comma and one space, and strings are surrounded by single quotes. Note the difference between the string representations of the following lists:
>
> ```python
> >>> num_list = [1, 2, 3]
> >>> str_list = ['1', '2', '3']
> >>> print(num_list)
> [1, 2, 3]
> >>> print(str_list)
> ['1', '2', '3']
> ```

Both methods are in the `LinkedList` class above (`size` is updated by `append()`, `remove()`, and `insert()`).

In [7]:
l = LinkedList()
for i in [1, 3, 5]:
    l.append(i)
print(len(l))
l.append(7)
print(len(l))

l1, l2, l3 = LinkedList(), LinkedList(), LinkedList()
for i in [1, 3, 5]:
    l1.append(i)
for i in ['a', 'b', "c"]:
    l2.append(i)
for i in ['1', '2', '3']:
    l3.append(i)
print(l1, l2, l3, LinkedList(), sep="\n")
for items in [[1, 3, 5], ['a', 'b', 'c'], ['1', '2', '3'], [], [1.5, 'x', 2]]:
    lst = LinkedList()
    for x in items:
        lst.append(x)
    assert str(lst) == str(items) and len(lst) == len(items)

3
4
[1, 3, 5]
['a', 'b', 'c']
['1', '2', '3']
[]


## remove()

To delete a node, every reference to it must be removed; Python then deletes the object automatically, since there is no way for the user to access it. Naïvely severing the links into the target node also severs the rest of the chain:

```python
def remove(self, data):
    """Attempt to remove the first node containing 'data'.
    This method incorrectly removes additional nodes.
    """
    # Find the target node and sever the links pointing to it.
    target = self.find(data)
    target.prev.next = None         # -/-> target
    target.next.prev = None         # target <-/-
```

With this version, removing 4 from `[0, 1, ..., 9]` leaves `[0, 1, 2, 3]`. The fix is to point the previous node's `next` at the node after the target, and that node's `prev` at the node before it, so the target is unreachable but the chain stays connected.

## Problem 4: remove()

> **Problem 4.** Modify the `remove()` method given above so that it correctly removes the first node in the list containing the specified data. Account for the special cases of removing the first, last, or only node.

> **Achtung!** Python keeps track of the variables in use and automatically deletes a variable if there is no access to it. In many other languages, leaving a reference to an object without explicitly deleting it could cause a serious memory leak. See https://docs.python.org/2/library/gc.html for more information on Python's auto-cleanup system.

In [8]:
def build(items):
    lst = LinkedList()
    for x in items:
        lst.append(x)
    return lst

def backward(lst):
    """Read the list tail-to-head through the prev links."""
    out, node = [], lst.tail
    while node is not None:
        out.append(node.value)
        node = node.prev
    return out[::-1]

my_list = build(range(10))
my_list.remove(4)
print(my_list)                  # [0, 1, 2, 3, 5, 6, 7, 8, 9]

l1 = build([1, 3, 5, 7, 9])
l1.remove(5)                    # middle
l1.remove(1)                    # head
l1.remove(9)                    # tail
print(l1, len(l1))
assert str(l1) == "[3, 7]" and backward(l1) == [3, 7] and l1.head.prev is None and l1.tail.next is None

only = build([10])
only.remove(10)                 # only node
print(only, len(only), only.head, only.tail)

for lst, target in [(build([2, 4, 6, 8]), 10), (LinkedList(), 10)]:
    try:
        lst.remove(target)
    except ValueError as e:
        print(f"remove({target}) from {lst} -> ValueError: {e}")
    else:
        raise AssertionError("remove() should raise ValueError")

[0, 1, 2, 3, 5, 6, 7, 8, 9]
[3, 7] 2
[] 0 None None
remove(10) from [2, 4, 6, 8] -> ValueError: Data doesn't exist.
remove(10) from [] -> ValueError: The list is empty.


## Problem 5: insert()

> **Problem 5.** Add a method called `insert(self, data, place)` to the `LinkedList` class that inserts a new node containing `data` immediately before the first node in the list containing `place`. Account for the special case of inserting before the first node.
>
> Note that since `insert()` places a new node before an existing node, it is not possible to use `insert()` to put a new node at the end of the list or in an empty list (use `append()` instead).

> **Note.** The temporal complexity for inserting to the beginning or end of a linked list is $O(1)$, but inserting anywhere else is $O(n)$, where $n$ is the number of nodes in the list. This is quite slow compared other data structures. In the next lab we turn our attention to trees, special kinds of linked lists that allow for much quicker sorting and data retrieval.

In [9]:
l1 = build([1, 3, 7])
l1.insert(7, 7)
print(l1)                       # [1, 3, 7, 7]
l1.insert(5, 7)
print(l1)                       # [1, 3, 5, 7, 7]
l1.insert(0, 1)                 # before the head
print(l1, len(l1))
assert str(l1) == "[0, 1, 3, 5, 7, 7]" and backward(l1) == [0, 1, 3, 5, 7, 7] and l1.head.value == 0

for lst, args in [(l1, (3, 2)), (LinkedList(), (10, 10))]:
    try:
        lst.insert(*args)
    except ValueError as e:
        print(f"insert{args} into {lst} -> ValueError: {e}")
    else:
        raise AssertionError("insert() should raise ValueError")

[1, 3, 7, 7]
[1, 3, 5, 7, 7]
[0, 1, 3, 5, 7, 7] 6
insert(3, 2) into [0, 1, 3, 5, 7, 7] -> ValueError: Data doesn't exist.
insert(10, 10) into [] -> ValueError: The list is empty.


## Restricted-Access Lists

It is sometimes wise to restrict the user's access to some of the data within a structure. The three most common restricted-access structures are:

- **Stack:** Last In, First Out (LIFO), like a pile of plates. Usually `push()` inserts new data and `pop()` removes and returns the last piece of data inserted.
- **Queue** (pronounced "cue"): First In, First Out (FIFO), like a line at the bank. `push()` inserts data at the end of the queue and `pop()` removes and returns the data at the front (these are sometimes called `enqueue()` and `dequeue()`).
- **Deque** (pronounced "deck"): a double-ended queue, like a deck of cards. Data can be inserted or removed from either end, but data in the middle is inaccessible. A deque has `append()`, `appendleft()`, `pop()`, and `popleft()`.

## Problem 6: A Deque class

> **Problem 6.** Write a `Deque` class that inherits from the `LinkedList` class.
>
> 1. Use inheritance to implement the following methods:
>    - `pop(self)`: Remove the last node in the list and return its data.
>    - `popleft(self)`: Remove the first node in the list and return its data.
>    - `appendleft(self, data)`: Insert a new node containing `data` at the beginning of the list.
>
>    The `LinkedList` class already implements the `append()` method.
> 2. Override the `remove()` method with the following:
>
>    ```python
>    def remove(*args, **kwargs):
>        raise NotImplementedError("Use pop() or popleft() for removal")
>    ```
>
>    This effectively disables `remove()` for the `Deque` class, preventing the user from removing a node from the middle of the list.
> 3. Disable the `insert()` method as well.

> **Note.** The `*args` argument allows the `remove()` method to receive any number of positional arguments without raising a `TypeError`, and the `**kwargs` argument allows it to receive any number of keyword arguments. This is the most general form of a function signature.

In [10]:
# Problem 6: Write a Deque class.
class Deque(LinkedList):

    def __init__(self):
        LinkedList.__init__(self)

    def pop(self):
        if self.size == 1:
            old_tail = self.tail
            self.tail = None
            self.head = None
            self.size -= 1
            return old_tail.value

        old_tail = self.tail
        temp = self.tail.prev
        temp.next = None
        self.tail = temp
        self.size -= 1
        return old_tail.value

    def popleft(self):
        if self.size == 1:
            old_head = self.head
            self.tail = None
            self.head = None
            self.size -= 1
            return old_head.value

        old_head = self.head
        temp = self.head.next
        temp.prev = None
        self.head = temp
        self.size -= 1
        return old_head.value

    def appendleft(self, data):
        new_node = LinkedListNode(data)
        if self.head is None:
            # If the list is empty, assign the head and tail attributes to
            # new_node, since it becomes the first and last node in the list.
            self.head = new_node
            self.tail = new_node
            self.size += 1
        else:
            self.head.prev = new_node      
            new_node.next = self.head
            self.head = new_node
            self.size += 1

    def remove(*args, **kwargs):
        raise NotImplementedError("Use pop() or popleft() for removal")

    def insert(*args, **kwargs):
        raise NotImplementedError("Use append() or appendleft() for insertion")

In [11]:
d = Deque()
for i in [3, 4, 5]:
    d.append(i)
for i in [2, 1]:
    d.appendleft(i)
print(d, len(d))
print(d.pop(), d.popleft(), d)
assert str(d) == "[2, 3, 4]" and backward(d) == [2, 3, 4]

single = Deque()
single.append(47)
print(single.pop(), single, len(single))         # popping the only node
single.appendleft(41)
print(single.popleft(), single, len(single))
assert single.head is None and single.tail is None

for method, args in [(d.remove, (3,)), (d.insert, (0, 3))]:
    try:
        method(*args)
    except NotImplementedError as e:
        print(f"{method.__name__} -> NotImplementedError: {e}")

[1, 2, 3, 4, 5] 5
5 1 [2, 3, 4]
47 [] 0
41 [] 0
remove -> NotImplementedError: Use pop() or popleft() for removal
insert -> NotImplementedError: Use append() or appendleft() for insertion


Python lists have `append()` and `pop()` methods, so they can be used as stacks, but access and removal at the front are much slower because Python lists are not implemented as linked lists. The `collections` module in the standard library has a `deque` object, implemented as a doubly linked list, which is an excellent choice in practice when speed is of the essence and data only needs to be accessed from the ends of the list.

## Problem 7: Reversing a file

> **Problem 7.** Write a function that accepts the name of a file to be read and a file to write to. Read the first file, adding each line to the end of a deque. After reading the entire file, pop each entry off of the end of the deque one at a time, writing the result to a line of the second file.
>
> For example, if the file to be read has the list of words on the left, the resulting file should have the list of words on the right.
>
> | File to read | Resulting file |
> |---|---|
> | My homework is too hard for me. | I am a mathematician. |
> | I do not believe that | Programming is hard, but |
> | I can solve these problems. | I can solve these problems. |
> | Programming is hard, but | I do not believe that |
> | I am a mathematician. | My homework is too hard for me. |
>
> You may use a Python list, your `Deque` class, or `collections.deque` for the deque. Test your function on the file `english.txt`, which contains a list of over 58,000 English words in alphabetical order.

In [12]:
# Problem 7
def prob7(infile, outfile):
    """Reverse the file 'infile' by line and write the results to 'outfile'."""
    lines = Deque()
    try:
        with open(infile, 'r') as myfile:
            for line in myfile:
                lines.append(line.strip('\n'))

    except (FileNotFoundError, TypeError, OSError) as e:
        print("Enter a valid file name.")

    with open(outfile, 'w') as outfile:
        for i in range(len(lines)):
            outfile.write(lines.pop() + '\n')

In [13]:
prob7("english.txt", "out.txt")

with open("english.txt") as f:
    original = f.read().splitlines()
with open("out.txt") as f:
    reversed_lines = f.read().splitlines()

print(f"{len(original)} lines in, {len(reversed_lines)} lines out")
print("english.txt (first 5):", original[:5])
print("out.txt     (first 5):", reversed_lines[:5])
assert reversed_lines == original[::-1]

58108 lines in, 58108 lines out
english.txt (first 5): ['aardvark', 'aardwolf', 'aaron', 'aback', 'abacus']
out.txt     (first 5): ['zulu', 'zoos', 'zooplankton', 'zooms', 'zooming']


## Additional Material

Ideas from the spec for extending the linked list:

- **Constructor from an iterable:** a keyword argument so that `LinkedList([1, 2, 3, 4, 5])` casts an iterable as a linked list, the way iterables can be cast as Python's standard structures.
- **More methods:** `count()` (occurrences of a value), and in-place `reverse()`, `rotate()` (a given number of steps to the right), and `sort()`.
- **More magic methods:** `__add__()` to concatenate two lists, `__getitem__()` and `__setitem__()` for bracket indexing, and `__iter__()` for `for` loops, `iter()`, and the `in` statement.
- **Sorted list:** a `SortedLinkedList` subclass whose `add(self, data)` inserts `data` before the first node with a value greater than or equal to it, with the other ways of adding nodes disabled. A linked list is not an ideal implementation for a sorted list (try sorting `english.txt`).
- **Circular linked list:** the "last" node connects back to the "first" node, so a reference to the tail is unnecessary.